In [1]:
import sys
import torch
import importlib
import torch.nn as nn
from peft import LoraConfig, get_peft_model

sys.path.append("/home/yohan.abeysinghe/Pangu/pangu-pytorch")

from models.pangu_model import PanguModel

config_module = importlib.import_module(f"configs.{'config3'}")
cfg = config_module.cfg

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

/l/users/yohan.abeysinghe/miniconda3/envs/pangu/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [2]:
model = PanguModel(device=device, cfg=cfg).to(device)

In [3]:
checkpoint = torch.load(cfg.PG.BENCHMARK.PRETRAIN_24_torch, weights_only=False, map_location='cuda')
state_dict = checkpoint['model']

In [4]:
# Learning rate for new variables.
model_state_dict = model.state_dict()

# Modify input layer for dimension matching and loading the existing weights
# for first 112 channels. Rest is initialized randomly.
new_input_weight = torch.zeros((192, 160, 1))
new_input_weight[:, :112, :] = state_dict['_input_layer.conv_surface.weight']
nn.init.xavier_uniform_(new_input_weight[:, 112:, :])
state_dict['_input_layer.conv_surface.weight'] = new_input_weight

# Modify output layer for dimension matching and loading the existing weights
# for first 64 channels. Rest is initialized randomly.
new_output_weight = torch.zeros((112, 384, 1))
new_output_weight[:64, :, :] = state_dict['_output_layer.conv_surface.weight']
nn.init.xavier_uniform_(new_output_weight[64:, :, :])
state_dict['_output_layer.conv_surface.weight'] = new_output_weight

# Modify output layer bias. Loading first 64 biases.
new_output_bias = torch.zeros(112)
new_output_bias[:64] = state_dict['_output_layer.conv_surface.bias']
state_dict['_output_layer.conv_surface.bias'] = new_output_bias

In [5]:
model.load_state_dict(state_dict, strict=False)

<All keys matched successfully>

In [28]:
target_modules = []

for n, m in model.named_modules():
    if isinstance(m, nn.Linear):
        target_modules.append(n)
        print(f"appended {n}")

config = LoraConfig(
    r=cfg.PG.TRAIN.Low_Rank,
    lora_alpha=16,
    target_modules=target_modules,
    lora_dropout=0.1,
    # modules_to_save=["_output_layer.conv_surface","_output_layer.conv"]
)

peft_model = get_peft_model(model, config)

appended downsample.linear
appended layers.EarthSpecificLayer0.blocks.EarthSpecificBlock0.linear.linear1
appended layers.EarthSpecificLayer0.blocks.EarthSpecificBlock0.linear.linear2
appended layers.EarthSpecificLayer0.blocks.EarthSpecificBlock0.attention.linear1
appended layers.EarthSpecificLayer0.blocks.EarthSpecificBlock0.attention.linear2
appended layers.EarthSpecificLayer0.blocks.EarthSpecificBlock1.linear.linear1
appended layers.EarthSpecificLayer0.blocks.EarthSpecificBlock1.linear.linear2
appended layers.EarthSpecificLayer0.blocks.EarthSpecificBlock1.attention.linear1
appended layers.EarthSpecificLayer0.blocks.EarthSpecificBlock1.attention.linear2
appended layers.EarthSpecificLayer1.blocks.EarthSpecificBlock0.linear.linear1
appended layers.EarthSpecificLayer1.blocks.EarthSpecificBlock0.linear.linear2
appended layers.EarthSpecificLayer1.blocks.EarthSpecificBlock0.attention.linear1
appended layers.EarthSpecificLayer1.blocks.EarthSpecificBlock0.attention.linear2
appended layers.Ear

In [29]:
for param in model._input_layer.conv_surface.parameters():
    param.requires_grad = True
for param in model._output_layer.conv_surface.parameters():
    param.requires_grad = True